
# Batch PERCEIVE Export

Run the PERCEIVE pipeline over every video in `data/videos/` and export
per-frame scene, face, and fused results to JSON for both stabilizer
settings.


In [ ]:
# Setup: repository path and imports
import sys
import json
import math
import torch.serialization
from fastai.learner import Learner

torch.serialization.add_safe_globals([Learner])
# then instantiate SceneCLIPAdapter as before

from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from utils.runtime_driver import perceive_video

print("Repo root:", repo_root)


In [ ]:
# Adapters (best effort)
scene_adapter = None
face_processor = None
face_adapter = None

try:
    from models.scene.clip_vit_scene_adapter import SceneCLIPAdapter

    scene_adapter = SceneCLIPAdapter(
        model_name="openai/clip-vit-base-patch32",
        dropout_rate=0.3,
        device="auto",
        tta=3,
        weights_path="scene/checkpoints/clip_vit-b32_improved_head.pth",
        auto_load_best=False,  # avoids falling back to the .pkl if this load fails
    )

    print("SceneCLIPAdapter: OK")
except Exception as exc:
    print(f"SceneCLIPAdapter unavailable: {exc}")

try:
    from utils.emonet_single_face_processor import EmoNetSingleFaceProcessor

    face_processor = EmoNetSingleFaceProcessor(
        min_detection_confidence=0.5, padding_ratio=0.2
    )
    if face_processor.available:
        print("EmoNetSingleFaceProcessor: OK")
    else:
        print("EmoNetSingleFaceProcessor: MediaPipe detector unavailable")
except Exception as exc:
    print(f"Face processor unavailable: {exc}")

try:
    from models.face.emonet_adapter import EmoNetAdapter

    face_adapter = EmoNetAdapter(
        ckpt_dir="models/emonet/pretrained",
        n_classes=8,
        device="auto",
        tta=3,
    )
    print("EmoNetAdapter: OK")
except Exception as exc:
    print(f"EmoNetAdapter unavailable: {exc}")


* `SCENE_TTA` – Scene adapter MC-dropout sample count per frame; higher (e.g., 5) means more stochastic passes, stronger smoothing, and richer variance estimates at higher compute cost, while lower (e.g., 1) is fast but collapses scene variance and leaves predictions noisier.
* `FACE_TTA` – Number of augmented face crops per frame (original + flip/jitter); increasing (e.g., 5) averages more variants for steadier face scores and non-zero variance but costs time, decreasing (e.g., 1) reverts to a single deterministic pass that is quicker yet more jittery with little uncertainty signal.
* `TARGET_SAMPLE_FPS` – Desired frame sampling rate; raising captures finer temporal detail and more compute, lowering skips nuance to save time.
* `MAX_FRAMES` – Hard cap on frames processed; enforcing a limit bounds runtime/memory but may miss long-range context, None processes entire clips.
* `USE_VARIANCE_WEIGHTING` – Enables inverse-variance fusion so consistent opinions dominate; disabling falls back to fixed weights, letting noisy paths influence the result equally.
* `SCENE_WEIGHT` – Fixed fallback weight for scene path when variance fusion can’t run; higher favors global context, lower reduces its voice.
* `FACE_WEIGHT` – Fixed fallback weight for face path; increasing leans into facial cues when variance data is missing, decreasing dampens them. Keep scene_weight + face_weight sensible.
* `FACE_SCORE_THRESHOLD` – Minimum face confidence; lifting it drops weak detections (less jitter but more gaps), lowering keeps borderline faces (more coverage, more noise).
* `FACE_MAX_SIGMA` – Ceiling on face uncertainty (sigma) before gating; tighter caps ignore very uncertain face outputs, looser/None leaves them in the mix.
* `BRIGHTNESS_THRESHOLD` – Luma floor for using face results; higher filters dim frames (steadier but risky in low light), lower keeps more frames (better coverage, noisier in dark scenes).


Stabilizer Settings:


* `UNCERTAINTY_THRESHOLD` – Variance level that triggers the stabilizer’s hold; lower values freeze outputs more often, higher values let raw updates through unless they’re very uncertain.
* `STABILIZER_WINDOW` – Rolling window for jitter/variance metrics; longer windows give smoother but laggier stability stats, shorter windows react faster with more fluctuation.
* `STABILIZER_ALPHA` – EMA blend factor; higher alpha tracks new values quickly (less smoothing), lower alpha leans on history for steadier but slower responses.


In [ ]:
# Configuration
VIDEO_DIR = repo_root / "data" / "VEATIC" / "videos"
OUTPUT_ROOT = repo_root / "results" / "inference"
VIDEO_EXTENSIONS = (".mp4", ".mov", ".mkv", ".avi")

SCENE_TTA = 3
FACE_TTA = 3
TARGET_SAMPLE_FPS = 1.0
MAX_FRAMES = None  # e.g., 600 to cap processing time
USE_VARIANCE_WEIGHTING = True
SCENE_WEIGHT = 0.6
FACE_WEIGHT = 0.4
FACE_SCORE_THRESHOLD = None
FACE_MAX_SIGMA = None
BRIGHTNESS_THRESHOLD = None

# Stabilizer parameters (used when enable_stabilizer=True)
UNCERTAINTY_THRESHOLD = 0.4
STABILIZER_WINDOW = 60
STABILIZER_ALPHA = 0.7

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Video directory:", VIDEO_DIR)
print("Output root:", OUTPUT_ROOT)


In [ ]:
# Helpers
from typing import List, Dict, Any


def list_videos() -> List[Path]:
    if not VIDEO_DIR.exists():
        return []
    videos: List[Path] = []
    for ext in VIDEO_EXTENSIONS:
        videos.extend(VIDEO_DIR.glob(f"*{ext}"))
    return sorted(videos)


def _clean_sequence(values: List[float]) -> List[Any]:
    cleaned: List[Any] = []
    for value in values:
        if isinstance(value, float):
            if math.isfinite(value):
                cleaned.append(float(value))
            else:
                cleaned.append(None)
        else:
            cleaned.append(value)
    return cleaned


def _coverage(series) -> float:
    total = len(series.valence)
    if total == 0:
        return 0.0
    valid = sum(1 for v in series.valence if isinstance(v, float) and math.isfinite(v))
    return valid / total


def series_to_dict(series, include_coverage: bool = False) -> Dict[str, Any]:
    data: Dict[str, Any] = {
        "valence": _clean_sequence(series.valence),
        "arousal": _clean_sequence(series.arousal),
        "var_valence": _clean_sequence(series.var_valence),
        "var_arousal": _clean_sequence(series.var_arousal),
        "mean_valence": series.mean_valence,
        "mean_arousal": series.mean_arousal,
        "median_valence": series.median_valence,
        "median_arousal": series.median_arousal,
        "mode_valence": series.mode_valence,
        "mode_arousal": series.mode_arousal,
    }
    if include_coverage:
        data["coverage"] = _coverage(series)
    return data


def build_payload(result, *, enable_stabilizer: bool) -> Dict[str, Any]:
    if result.fps and result.fps > 0:
        timestamps = [idx / result.fps for idx in result.frame_indices]
    else:
        timestamps = [None] * len(result.frame_indices)

    return {
        "video": {
            "path": str(result.video_path),
            "width": result.width,
            "height": result.height,
            "fps": result.fps,
            "frame_count": result.frame_count,
        },
        "processing": {
            "processed_frames": result.processed_frames,
            "sample_stride": result.sample_stride,
            "effective_fps": result.effective_fps,
            "enable_stabilizer": enable_stabilizer,
            "scene_tta": SCENE_TTA,
            "face_tta": FACE_TTA,
            "target_sample_fps": TARGET_SAMPLE_FPS,
            "max_frames": MAX_FRAMES,
            "use_variance_weighting": USE_VARIANCE_WEIGHTING,
            "scene_weight": SCENE_WEIGHT,
            "face_weight": FACE_WEIGHT,
            "uncertainty_threshold": UNCERTAINTY_THRESHOLD
            if enable_stabilizer
            else None,
            "stabilizer_window": STABILIZER_WINDOW if enable_stabilizer else None,
            "stabilizer_alpha": STABILIZER_ALPHA if enable_stabilizer else None,
        },
        "samples": {
            "frame_indices": list(result.frame_indices),
            "timestamps_sec": timestamps,
            "scene": series_to_dict(result.scene_result),
            "face": series_to_dict(result.face_result, include_coverage=True),
            "fusion": series_to_dict(result.fusion_result),
        },
    }


def export_batch(
    enable_stabilizer: bool, *, capture_overlays: bool = False
) -> List[Path]:
    videos = list_videos()
    label = "stabilized" if enable_stabilizer else "unstabilized"
    out_dir = OUTPUT_ROOT / label
    out_dir.mkdir(parents=True, exist_ok=True)

    if not videos:
        print(f"No videos found in {VIDEO_DIR}")
        return []

    saved: List[Path] = []
    for video_path in videos:
        print(f"[{label}] Processing {video_path.name} ...", end=" ")
        result = perceive_video(
            video_path,
            scene_predictor=scene_adapter,
            face_processor=face_processor,
            face_expert=face_adapter,
            scene_tta=SCENE_TTA,
            face_tta=FACE_TTA,
            target_sample_fps=TARGET_SAMPLE_FPS,
            max_frames=MAX_FRAMES,
            use_variance_weighting=USE_VARIANCE_WEIGHTING,
            scene_weight=SCENE_WEIGHT,
            face_weight=FACE_WEIGHT,
            face_score_threshold=FACE_SCORE_THRESHOLD,
            face_max_sigma=FACE_MAX_SIGMA,
            brightness_threshold=BRIGHTNESS_THRESHOLD,
            enable_stabilizer=enable_stabilizer,
            stabilizer_alpha=STABILIZER_ALPHA,
            uncertainty_threshold=UNCERTAINTY_THRESHOLD,
            stabilizer_window=STABILIZER_WINDOW,
            capture_overlays=capture_overlays,
            return_fusion=False,
        )

        payload = build_payload(result, enable_stabilizer=enable_stabilizer)
        dest = out_dir / f"{video_path.stem}.json"
        with dest.open("w", encoding="utf-8") as fh:
            json.dump(payload, fh, indent=2, ensure_ascii=False, allow_nan=False)
        saved.append(dest)
        print("done")

    return saved


In [ ]:
# Run batch export (stabilizer OFF)
unstabilized_outputs = export_batch(enable_stabilizer=False)
print(f"Unstabilized exports: {len(unstabilized_outputs)} files")


In [ ]:
# Run batch export (stabilizer ON)
stabilized_outputs = export_batch(enable_stabilizer=True)
print(f"Stabilized exports: {len(stabilized_outputs)} files")
